# Spectral clustering and the graph Laplacian

*maths · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/spectral_clustering)

Turn a cloud of points into a graph (Gaussian weights on $k$-nearest-neighbour edges), build the graph Laplacian $L = D - W$ and its normalised form $L_{sym}$ as tensors, and read the clusters off its lowest eigenvectors. Two moons and concentric rings, where plain k-means fails, are separated perfectly; the same Laplacian on a grid gives the vibration modes of a drum.

**Vector form**

$$L = D - W,\qquad L_{sym} = I - D^{-1/2} W D^{-1/2},\qquad \mathbf f^{\mathsf T} L\,\mathbf f = \tfrac12 \sum_{ij} W_{ij}(f_i - f_j)^2 \ge 0,\qquad L_{sym}\,\mathbf u_m = \lambda_m \mathbf u_m$$

**Index (tensor) form**

$$W_{ij} = e^{-(x_{ia}-x_{ja})(x_{ia}-x_{ja})/2\sigma^2}\,[ij \in \text{kNN}],\qquad d_i = W_{ij}1_j,\qquad L_{ij} = d_i\delta_{ij} - W_{ij},\qquad f_iL_{ij}f_j = \tfrac12 W_{ij}(f_i-f_j)^2$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{ia}$ | `points(name, n, noise, seed, dev)` | (N, 2) | point $i$, coordinate $a$: two moons, two rings or three blobs, with true labels for scoring |
| $(x_{ia}-x_{ja})(x_{ia}-x_{ja})$ | `sq[:, None] + sq[None, :] - 2 * einsum('ia,ja->ij', x, x)` | (N, N) | repeated $a$ ⇒ summed: all squared distances at once |
| $W_{ij}$ | `gaussian_knn_weights(d2, sigma, k)` | (N, N) | Gaussian similarity kept only if $j$ is one of the $k$ nearest neighbours of $i$ or the other way round; symmetric, zero diagonal |
| $d_i = W_{ij}1_j$ | `einsum('ij->i', w)` | (N,) | repeated $j$ ⇒ summed: the degree of each point |
| $L_{ij}$ | `torch.diag(degrees(w)) - w` | (N, N) | every row sums to 0, so the constant vector has eigenvalue 0 |
| $(L_{sym})_{ij}$ | `eye - einsum('i,ij,j->ij', s, w, s)` | (N, N) | $s_i = d_i^{-1/2}$; no sum, $i$ and $j$ stay free |
| $\lambda_m,\ u_{im}$ | `torch.linalg.eigh(L_sym)` | (N,), (N, N) | eigenvalues ascending; the first $k$ eigenvectors, rows scaled to unit length, are the spectral embedding |
| $u_{i1..k} \to \text{k-means}$ | `kmeans(u, k, seed)` | (N,) | Lloyd iterations with k-means++ starts; in the embedding the clusters are tight blobs |

Why it works: $f^{\mathsf T} L f = \frac12\sum_{ij} W_{ij}(f_i - f_j)^2$ is a sum of squares, so $L$ is positive semi-definite, and it is zero only if $f$ is constant across every edge, that is constant on each connected component. The number of zero eigenvalues is therefore exactly the number of components (compare the two counts in the terminal), and the indicator vectors of the components span that eigenspace. When clusters are joined by only a few weak edges, the eigenvalues are small instead of zero and the eigenvectors are close to indicators; minimising $f^{\mathsf T}Lf$ is a relaxed version of cutting the fewest edges, the normalised cut of Shi and Malik (2000). The second eigenvector (the Fiedler vector) gives the best two-way split. Plain k-means draws straight boundaries around centres, so it cuts the moons and rings in half; in the embedding (top row, watch the points slide) the clusters are separated blobs, and k-means works. The large gap in the spectrum after the $k$-th eigenvalue is a hint for the number of clusters. On a $g\times g$ grid graph $L = L_1\otimes I + I\otimes L_1$ is minus the discrete Laplacian with free (Neumann) edges, so its eigenvectors are the standing waves $\cos\frac{\pi a (i+\frac12)}{g}\cos\frac{\pi b (j+\frac12)}{g}$ with $\lambda = (2 - 2\cos\frac{\pi a}{g}) + (2 - 2\cos\frac{\pi b}{g})$: the same modes as the vibrating drum in the waves topic (which has clamped edges instead). Modes $(a, b)$ and $(b, a)$ share an eigenvalue, and eigh returns some mixture of the pair, which is why some look diagonal. Results depend on $\sigma$ and $k$: too few neighbours split a cluster into pieces, too many add bridges between clusters. Further reading: U. von Luxburg, A tutorial on spectral clustering (2007); Ng, Jordan and Weiss (2001).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/spectral_clustering/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import dirichlet_energy, gaussian_knn_weights, laplacian, squared_distances


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n = 150 if fast_mode() else params["n_points"]
    results = {}
    for name in tensors.DATASETS:
        x, truth = tensors.points(name, n, params["noise"], 0, dev)  # (N, 2), (N,)
        k = tensors.N_CLUSTERS[name]
        w = gaussian_knn_weights(squared_distances(x), params["sigma"], params["k_neighbours"])  # (N, N)
        lap = laplacian(w)
        evals_l, _ = solve.spectrum(lap)
        n_zero = solve.count_zero_eigenvalues(evals_l)
        n_comp = int(solve.connected_components(w).unique().numel())
        spectral, evals, embedding = solve.spectral_clustering(w, k, seed=0)
        plain = solve.kmeans(x, k, seed=0)
        ari_s, ari_k = solve.adjusted_rand_index(truth, spectral), solve.adjusted_rand_index(truth, plain)
        lam2, fiedler = solve.fiedler_vector(w)
        f = torch.randn(n, generator=torch.Generator().manual_seed(1), dtype=torch.float64).to(dev)
        print(f"{name}: N = {n}, {int((w > 0).sum()) // 2} edges, max |row sum of L| = {lap.sum(1).abs().max():.1e}")
        print(
            f"  smallest eigenvalue of L {evals_l[0]:+.1e} (≥ 0), zero eigenvalues {n_zero}, components {n_comp}; "
            f"f·Lf = {f @ lap @ f:.3f} = ½ΣW(fi−fj)² = {dirichlet_energy(w, f):.3f}"
        )
        print(f"  L_sym eigenvalues {[round(v, 4) for v in evals[: k + 2].tolist()]}, Fiedler value λ₂ = {lam2:.4f}")
        print(f"  adjusted Rand index: spectral {ari_s:.3f}, plain k-means {ari_k:.3f}")
        results[name] = {
            "x": x,
            "spectral": spectral,
            "kmeans": plain,
            "ari": (ari_s, ari_k),
            "evals": evals[:10],
            "fiedler": fiedler,
            "embedding": embedding,
            "w": w,
        }

    # vibration modes of a grid graph: L = L₁ ⊗ I + I ⊗ L₁ with eigenvalues (2 − 2cos πa/n) + (2 − 2cos πb/n)
    g = params["grid"]
    evals_g, modes = solve.spectrum(laplacian(tensors.grid_graph(g, dev)))
    a = torch.arange(g, dtype=torch.float64, device=dev)
    one_d = 2 - 2 * torch.cos(math.pi * a / g)
    exact = (one_d[:, None] + one_d[None, :]).flatten().sort().values
    print(f"grid {g}×{g}: max |λ − exact (2 − 2cos πa/n) + (2 − 2cos πb/n)| = {(evals_g - exact).abs().max():.1e}")
    n_modes = 12
    print(f"first grid eigenvalues {[round(v, 4) for v in evals_g[:6].tolist()]} (pairs = degenerate modes)")
    return plot.animate(results, modes[:, 1 : n_modes + 1], evals_g[1 : n_modes + 1], g, out_dir / "spectral.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/spectral_clustering/equation.py
r"""Mathematics → tensor translation.

Points x_{ia} (i = point, a = coordinate) become a weighted graph:

    D2_{ij} = (x_{ia} − x_{ja})(x_{ia} − x_{ja})               a summed: squared distances
    W_{ij}  = exp(−D2_{ij} / 2σ²)   on k-nearest-neighbour edges, 0 elsewhere (and on the diagonal)
    d_i     = W_{ij} 1_j                                         j summed: the degree of point i
    L_{ij}  = d_i δ_{ij} − W_{ij}                                graph Laplacian L = D − W
    (L_sym)_{ij} = δ_{ij} − d_i^{−1/2} W_{ij} d_j^{−1/2}         normalised Laplacian

The key identity, with i and j summed,

    f_i L_{ij} f_j = ½ W_{ij} (f_i − f_j)²  ≥ 0,

says L is positive semi-definite, and f^T L f = 0 exactly when f is constant on every connected piece of the graph.
So the number of zero eigenvalues of L is the number of connected components.
"""

import torch


def squared_distances(x: torch.Tensor) -> torch.Tensor:
    r"""D2_{ij} = x_{ia}x_{ia} + x_{ja}x_{ja} − 2 x_{ia}x_{ja}   (a summed).   (N, 2) → (N, N)"""
    sq = torch.einsum("ia,ia->i", x, x)
    return (sq[:, None] + sq[None, :] - 2 * torch.einsum("ia,ja->ij", x, x)).clamp_min(0)


def gaussian_knn_weights(d2: torch.Tensor, sigma: float, k: int) -> torch.Tensor:
    r"""W_{ij} = exp(−D2_{ij}/2σ²) if j is among the k nearest neighbours of i or i among those of j.   (N, N)"""
    n = d2.shape[0]
    near = torch.zeros_like(d2, dtype=torch.bool)
    nearest = d2.topk(k + 1, dim=1, largest=False).indices  # (N, k+1), includes the point itself
    near[torch.arange(n, device=d2.device)[:, None], nearest] = True
    near = near | near.T  # symmetric graph
    near.fill_diagonal_(False)
    return torch.where(near, torch.exp(-d2 / (2 * sigma**2)), 0.0)


def degrees(w: torch.Tensor) -> torch.Tensor:
    r"""d_i = W_{ij} 1_j   (j summed).   (N, N) → (N,)"""
    return torch.einsum("ij->i", w)


def laplacian(w: torch.Tensor) -> torch.Tensor:
    r"""L_{ij} = d_i δ_{ij} − W_{ij}.   (N, N)"""
    return torch.diag(degrees(w)) - w


def normalized_laplacian(w: torch.Tensor) -> torch.Tensor:
    r"""(L_sym)_{ij} = δ_{ij} − d_i^{−1/2} W_{ij} d_j^{−1/2}   (no sum: i and j appear on both sides).   (N, N)"""
    s = degrees(w).clamp_min(1e-12).rsqrt()
    return torch.eye(w.shape[0], dtype=w.dtype, device=w.device) - torch.einsum("i,ij,j->ij", s, w, s)


def dirichlet_energy(w: torch.Tensor, f: torch.Tensor) -> torch.Tensor:
    r"""½ W_{ij} (f_i − f_j)²   (i, j summed): equals f_i L_{ij} f_j and can never be negative.   f: (N,) → ()"""
    return 0.5 * torch.einsum("ij,ij->", w, (f[:, None] - f[None, :]) ** 2)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/spectral_clustering/tensors.py
"""Build the concrete tensors: three synthetic point clouds with true labels, and a 2D grid graph."""

import math

import torch

DATASETS = ["moons", "circles", "blobs"]
N_CLUSTERS = {"moons": 2, "circles": 2, "blobs": 3}


def points(name: str, n: int, noise: float, seed: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Point cloud x_{ia} of shape (N, 2), float64, and its true cluster labels (N,)."""
    gen = torch.Generator().manual_seed(seed)
    k = N_CLUSTERS[name]
    labels = torch.arange(n) % k
    if name == "moons":  # two interleaved half circles
        t = torch.rand(n, generator=gen, dtype=torch.float64) * math.pi
        upper = torch.stack([torch.cos(t), torch.sin(t)], 1)
        lower = torch.stack([1 - torch.cos(t), 0.5 - torch.sin(t)], 1)
        x = torch.where(labels[:, None] == 0, upper, lower)
    elif name == "circles":  # a ring inside a ring
        t = torch.rand(n, generator=gen, dtype=torch.float64) * 2 * math.pi
        radius = torch.where(labels == 0, 1.0, 0.45).double()
        x = radius[:, None] * torch.stack([torch.cos(t), torch.sin(t)], 1)
    else:  # three round blobs
        centres = torch.tensor([[0.0, 0.0], [2.2, 0.4], [0.9, 2.0]], dtype=torch.float64)
        x = centres[labels] + 4 * noise * torch.randn(n, 2, generator=gen, dtype=torch.float64)
    x = x + noise * torch.randn(n, 2, generator=gen, dtype=torch.float64)
    return x.to(device), labels.to(device)


def grid_graph(n: int, device: torch.device) -> torch.Tensor:
    """Adjacency of an n×n grid (4 neighbours, free edges): W = A₁ ⊗ I + I ⊗ A₁ with A₁ the path graph.   (n², n²)"""
    a1 = torch.diag(torch.ones(n - 1, dtype=torch.float64), 1)
    a1 = a1 + a1.T  # path graph 0 – 1 – … – (n−1)
    eye = torch.eye(n, dtype=torch.float64)
    return (torch.kron(a1, eye) + torch.kron(eye, a1)).to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/spectral_clustering/solve.py
"""Eigendecomposition of the Laplacian, k-means, spectral clustering and the adjusted Rand index."""

import torch

from .equation import degrees, normalized_laplacian


def spectrum(lap: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigenvalues in ascending order (N,) and orthonormal eigenvectors in the columns (N, N)."""
    return torch.linalg.eigh(lap)


def count_zero_eigenvalues(evals: torch.Tensor, tol: float = 1e-8) -> int:
    return int((evals.abs() < tol).sum())


def connected_components(w: torch.Tensor) -> torch.Tensor:
    """Component label of every node by label propagation: each node takes the smallest label among its neighbours
    until nothing changes. Independent of the eigenvalues, so it checks them.   (N, N) → (N,)"""
    n = w.shape[0]
    linked = (w > 0) | torch.eye(n, dtype=torch.bool, device=w.device)
    label = torch.arange(n, device=w.device, dtype=torch.float64)
    while True:
        new = torch.where(linked, label[None, :], torch.inf).min(dim=1).values
        if torch.equal(new, label):
            return label.long()
        label = new


def kmeans(x: torch.Tensor, k: int, seed: int, n_init: int = 10, n_iter: int = 100) -> torch.Tensor:
    """Lloyd's algorithm with k-means++ starts; keeps the run with the smallest within-cluster sum of squares."""
    gen = torch.Generator().manual_seed(seed)
    best, best_cost = None, float("inf")
    for _ in range(n_init):
        centres = x[torch.randint(len(x), (1,), generator=gen)]
        for _ in range(k - 1):  # k-means++: next centre far from the current ones
            d2 = torch.cdist(x, centres).min(dim=1).values ** 2
            pick = torch.multinomial((d2 / d2.sum()).cpu(), 1, generator=gen)
            centres = torch.cat([centres, x[pick]])
        for _ in range(n_iter):
            label = torch.cdist(x, centres).argmin(dim=1)  # (N,)
            one_hot = torch.nn.functional.one_hot(label, k).to(x.dtype)  # (N, k)
            counts = one_hot.sum(0).clamp_min(1)
            new = torch.einsum("ik,ia->ka", one_hot, x) / counts[:, None]  # cluster means
            if torch.allclose(new, centres):
                break
            centres = new
        cost = (torch.cdist(x, centres).min(dim=1).values ** 2).sum().item()
        if cost < best_cost:
            best, best_cost = label, cost
    return best


def spectral_embedding(w: torch.Tensor, k: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Ng–Jordan–Weiss: the k lowest eigenvectors of L_sym with every row scaled to unit length.
    Returns the eigenvalues (N,) and the embedding (N, k)."""
    evals, evecs = spectrum(normalized_laplacian(w))
    u = evecs[:, :k]
    return evals, u / u.norm(dim=1, keepdim=True).clamp_min(1e-12)


def spectral_clustering(w: torch.Tensor, k: int, seed: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Eigenvectors → k-means in the embedding. Returns labels (N,), eigenvalues of L_sym (N,) and embedding (N, k)."""
    evals, u = spectral_embedding(w, k)
    return kmeans(u, k, seed), evals, u


def fiedler_vector(w: torch.Tensor) -> tuple[float, torch.Tensor]:
    """Second-smallest eigenpair of the random-walk Laplacian D⁻¹L, from L_sym: f = D^{-1/2} u₂."""
    evals, evecs = spectrum(normalized_laplacian(w))
    f = evecs[:, 1] * degrees(w).clamp_min(1e-12).rsqrt()
    return evals[1].item(), f / f.abs().max()


def adjusted_rand_index(a: torch.Tensor, b: torch.Tensor) -> float:
    """ARI (Hubert & Arabie): 1 for identical partitions up to renaming, ≈ 0 for random ones."""
    table = torch.einsum("ia,ib->ab", torch.nn.functional.one_hot(a).double(), torch.nn.functional.one_hot(b).double())

    def pairs(m: torch.Tensor) -> torch.Tensor:
        return (m * (m - 1) / 2).sum()

    n_pairs = pairs(torch.tensor(float(len(a)), dtype=torch.float64))
    index, rows, cols = pairs(table), pairs(table.sum(1)), pairs(table.sum(0))
    expected = rows * cols / n_pairs
    return ((index - expected) / (0.5 * (rows + cols) - expected)).item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/spectral_clustering/plot.py
"""Animate the points sliding into their spectral embedding (top), k-means for comparison (bottom),
the spectra and the vibration modes of a grid graph."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

N_FRAMES = 60


def morph_weight(f: int) -> float:
    """0 = data space, 1 = spectral embedding: hold, slide over, hold, slide back."""
    t = (f % N_FRAMES) / N_FRAMES
    ramp = np.clip((t - 0.1) / 0.3, 0, 1) if t < 0.55 else np.clip(1 - (t - 0.65) / 0.3, 0, 1)
    return float(3 * ramp**2 - 2 * ramp**3)


def to_box(u: np.ndarray, x: np.ndarray) -> np.ndarray:
    """Rescale the first two embedding coordinates into the bounding box of the data, so both fit one axis."""
    u = u[:, :2]
    lo, hi = u.min(0), u.max(0)
    return x.min(0) + (u - lo) / np.maximum(hi - lo, 1e-9) * (x.max(0) - x.min(0))


def animate(results: dict, modes: torch.Tensor, mode_evals: torch.Tensor, g: int, path: Path) -> Path:
    fig, axes = plt.subplots(2, 4, figsize=(10.5, 5.2))
    colours = np.array(ACCENT[:3])
    moving = {}
    for col, (name, r) in enumerate(results.items()):
        x = r["x"].cpu().numpy()
        target = to_box(r["embedding"].cpu().numpy(), x)
        ax_s, ax_k = axes[0, col], axes[1, col]
        dots = ax_s.scatter(x[:, 0], x[:, 1], s=7, c=colours[r["spectral"].cpu().numpy()])
        moving[name] = (dots, x, target)
        ax_s.set_title(f"{name}: spectral, ARI {r['ari'][0]:.2f}", fontsize=9)
        ax_k.scatter(x[:, 0], x[:, 1], s=7, c=colours[r["kmeans"].cpu().numpy()])
        ax_k.set_title(f"{name}: plain k-means, ARI {r['ari'][1]:.2f}", fontsize=9)
        for ax in (ax_s, ax_k):
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_aspect("equal", adjustable="datalim")
    label = fig.text(0.37, 0.965, "", ha="center", fontsize=10, color="#374151")

    ax_f = axes[0, 3]  # Fiedler vector of the moons graph, with the spectra as an inset
    moons = results["moons"]
    xm, fm = moons["x"].cpu().numpy(), moons["fiedler"].cpu().numpy()
    ax_f.scatter(xm[:, 0], xm[:, 1], s=7, c=fm, cmap="RdBu_r", vmin=-1, vmax=1)
    ax_f.set_title("moons: Fiedler vector $f_2$ (sign = cut)", fontsize=9)
    ax_f.set_xticks([])
    ax_f.set_yticks([])
    ax_e = ax_f.inset_axes((0.6, 0.6, 0.37, 0.28))
    for col, r in enumerate(results.values()):
        ev = r["evals"].cpu().numpy()
        ax_e.plot(np.arange(1, len(ev) + 1), ev, "o-", ms=2, lw=0.8, color=ACCENT[col])
    ax_e.set_title(r"$\lambda_m(L_{sym})$, 3 sets", fontsize=7, pad=2)
    ax_e.tick_params(labelsize=5, length=2)

    ax_g = axes[1, 3]
    shapes = modes.T.reshape(-1, g, g).cpu().numpy()  # (M, g, g)
    vmax = np.abs(shapes).max()
    img = ax_g.imshow(shapes[0], cmap=plt.get_cmap("RdBu_r", 15), vmin=-vmax, vmax=vmax)
    ax_g.set_xticks([])
    ax_g.set_yticks([])
    ax_g.grid(False)
    title_g = ax_g.set_title("", fontsize=9)
    per_mode = N_FRAMES / len(shapes)

    def update(f: int):
        s = morph_weight(f)
        for dots, x, target in moving.values():
            dots.set_offsets((1 - s) * x + s * target)
        label.set_text(f"top: data space → spectral embedding (weight {s:.2f}), coloured by spectral clusters")
        m = min(int(f / per_mode), len(shapes) - 1)
        phase = (f - m * per_mode) / per_mode
        img.set_data(shapes[m] * np.cos(2 * np.pi * phase))
        title_g.set_text(f"grid mode {m + 2}, $\\lambda$ = {mode_evals[m]:.3f}")

    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return save_gif(fig, update, frame_indices(N_FRAMES, N_FRAMES), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_points': 300,
    'noise': 0.08,
    'k_neighbours': 10,
    'sigma': 0.2,
    'grid': 24
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.spectral_clustering")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.spectral_clustering.main")
problem_dir = Path("problems/maths/spectral_clustering")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
